# Libs & Setup

In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import logging
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import talib as ta
import math
from torch import optim
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split
from tqdm import tqdm
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.optim import AdamW
from utils import inspect, inverse_transform, monte_carlo_statistic, calc_expected_returns, calc_covariance, calc_volatility
from utils.paths import CHECKPOINTS_DIR, REPORTS_SIM_DIR, RESULTS_DIR, REPORTS_QS_DIR
from pypfopt import risk_models, expected_returns, plotting, EfficientFrontier

import optuna
from optuna.trial import TrialState
import os
import random

# Own Libs
from config import *
from entities import *
from strategies import *
from datasets import *
from engine import Engine
from models import DiffusionTransformer, Diffusion

/home/narodom.y@FUSION.LAB/.conda/envs/tsgenai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# logging.basicConfig(level=logging.DEBUG)
logging.getLogger('matplotlib').setLevel(logging.WARNING)

## Random Seed

In [3]:
def set_seed(seed: int = 78):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # สำหรับ GPU
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(78)

In [4]:
# cfg = TrainConfig(epochs=2, window_size=64, device=torch.device("cuda:1"))


device = torch.device("cuda:0") # if torch.cuda.is_available() else "cpu")
batch_size = 64
batch_size_exp = 1
epochs = 1000

# Window
window_size = 30
stride = 1

# Simulation
steps_sim = 60
paths_sim = 1000

# Optimizer
lr: float = 1e-4
weight_decay: float = 0.01 # 1e-6
betas: tuple = (0.9, 0.999)
eps: float = 1e-8

# Optuna
n_trials = 100 
min_resource = 3
max_resource = 50
reduction_factor = 3
EPOCHS_PER_TRIAL = 50


# Model
ddpm = {
    'timesteps': 1000, # 50 - 200
    'beta_start': 0.00020095977802093984,
    'beta_end': 0.029157631381838023
}

ddpm_transformer = {
    'window_size': window_size,
    'd_model': 512,
    'nhead': 8,
    'num_layers': 3,
    'dim_feedforward': 1536,
    'dropout': 0.1
}


# Data
time_range = {
    'start_date': '2021-01-01',
    'end_date': '2024-12-31'
}

# Indicator & condition
time_prd = 20
day_shift = 1

# N channels
n_prices = 1
n_targets = 1
n_conditions = 12


ckpt_name = f"ddpm_transformer_d{ddpm_transformer['d_model']}_l{ddpm_transformer['num_layers']}"
checkpoint_dir = os.path.join(RESULTS_DIR,"experimental_checkpoints_lab")
checkpoint_filename = f"tr{n_trials}_d{ddpm_transformer['d_model']}_dff{ddpm_transformer['dim_feedforward']}_l{ddpm_transformer['num_layers']}_h{ddpm_transformer['nhead']}_t{ddpm['timesteps']}.pt"

# Data

In [5]:
def time_range_info(df):
    info = (df.index.min(), df.index.max())
    print(f"Data range: {info[0]} to {info[1]}")
    
    duration = df.index.max() - df.index.min()
    print(f"Total duration: {duration}")

def time_range_mask(df, start_date, end_date):
    mask = (df.index >= start_date) & (df.index <= end_date)
    return mask

In [6]:
symbols = ['AAPL', 'TSLA', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'GOOG', 'META', 'AVGO', 'ORCL', 'CRM', 'ADBE', 'AMD', 'CSCO']
freq = "1d"

# Basket
basket = Basket(symbols=symbols)
basket.load_all_assets(freq=freq)

print(f"Basket data shape: {basket.data.shape}")
basket.data.head(5)

Basket data shape: (2760, 70)


AAPL                                                \
                Close       High        Low       Open       Volume   
Date                                                                  
2015-01-02  24.261047  24.729270  23.821672  24.718174  212818400.0   
2015-01-05  23.577574  24.110150  23.391173  24.030263  257142000.0   
2015-01-06  23.579794  23.839424  23.218085  23.641928  263188400.0   
2015-01-07  23.910433  24.010290  23.677430  23.788384  160423600.0   
2015-01-08  24.829119  24.886815  24.121236  24.238848  237458000.0   

                 TSLA                                             ...   AMD  \
                Close       High        Low       Open    Volume  ... Close   
Date                                                              ...         
2015-01-02  14.620667  14.883333  14.217333  14.858000  71466000  ...  2.67   
2015-01-05  14.006000  14.433333  13.810667  14.303333  80527500  ...  2.66   
2015-01-06  14.085333  14.280000  13.614000  14.004000  93928500  ...  2.63   
2015-01-07  14.063333  14.318667  13.985333  14.223333  44526000  ...  2.58   
2015-01-08  14.041333  14.253333  14.000667  14.187333  51637500  ...  2.61   

                                               CSCO                        \
            High   Low  Open      Volume      Close       High        Low   
Date                                                                        
2015-01-02  2.67  2.67  2.67         0.0  19.815605  20.181631  19.650534   
2015-01-05  2.70  2.64  2.67   8878200.0  19.420874  19.700776  19.377812   
2015-01-06  2.66  2.55  2.65  13912500.0  19.413698  19.865848  19.406522   
2015-01-07  2.65  2.54  2.63  12377600.0  19.593126  19.664896  19.363463   
2015-01-08  2.65  2.56  2.59  11136600.0  19.743843  20.160107  19.715135   

                                   
                 Open      Volume  
Date                               
2015-01-02  19.995029  22926500.0  
2015-01-05  19.607475  29460600.0  
2015-01-06  19.478291  47297600.0  
2015-01-07  19.478295  27570800.0  
2015-01-08  19.765374  40907000.0  

[5 rows x 70 columns]

In [7]:
time_range_info(basket.data)

for symbol, asset in basket.assets.items():
    mask = time_range_mask(asset.data, time_range['start_date'], time_range['end_date'])
    asset.data = asset.data[mask]

time_range_info(basket.data)

Data range: 2015-01-02 00:00:00 to 2025-12-22 00:00:00
Total duration: 4007 days 00:00:00
Data range: 2021-01-04 00:00:00 to 2024-12-31 00:00:00
Total duration: 1457 days 00:00:00


In [8]:
targets = ["Close"]
features = basket.get_unique_features()
print(f"Features:\t{features}\nTargets:\t{targets}")

Features:	['Close', 'High', 'Low', 'Open', 'Volume']
Targets:	['Close']


In [9]:
df = basket.data
df.ffill(inplace=True)
df.head()

AAPL                                                 \
                 Close        High         Low        Open     Volume   
Date                                                                    
2021-01-04  126.096588  130.189048  123.514437  130.101356  143301900   
2021-01-05  127.655609  128.366929  125.141666  125.589894   97664900   
2021-01-06  123.358521  127.694587  123.144152  124.449847  155088000   
2021-01-07  127.567940  128.259768  124.586290  125.073488  109578200   
2021-01-08  128.668976  129.234127  126.895568  129.039236  105158200   

                  TSLA                                                 ...  \
                 Close        High         Low        Open     Volume  ...   
Date                                                                   ...   
2021-01-04  243.256668  248.163330  239.063339  239.820007  145914600  ...   
2021-01-05  245.036667  246.946671  239.733337  241.220001   96735600  ...   
2021-01-06  251.993332  258.000000  249.699997  252.830002  134100000  ...   
2021-01-07  272.013336  272.329987  258.399994  259.209991  154496700  ...   
2021-01-08  293.339996  294.829987  279.463318  285.333344  225166500  ...   

                  AMD                                                  CSCO  \
                Close       High        Low       Open    Volume      Close   
Date                                                                          
2021-01-04  92.300003  96.059998  90.919998  92.110001  51802600  38.239326   
2021-01-05  92.769997  93.209999  91.410004  92.099998  34208000  38.256729   
2021-01-06  90.330002  92.279999  89.459999  91.620003  51911700  38.622074   
2021-01-07  95.160004  95.510002  91.199997  91.330002  42897200  39.109196   
2021-01-08  94.580002  96.400002  93.269997  95.980003  39816400  39.196186   

                                                       
                 High        Low       Open    Volume  
Date                                                   
2021-01-04  38.595972  37.708707  38.543782  24392500  
2021-01-05  38.335017  37.734811  37.995770  17763700  
2021-01-06  39.030909  38.178440  38.387210  21823100  
2021-01-07  39.239677  38.422000  38.448099  18218800  
2021-01-08  39.500638  38.491593  38.691662  20936300  

[5 rows x 70 columns]

In [10]:
assets = df.columns.get_level_values(0).unique()
assets

Index(['AAPL', 'TSLA', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'GOOG', 'META', 'AVGO',
       'ORCL', 'CRM', 'ADBE', 'AMD', 'CSCO'],
      dtype='object')

# Feature Engineering & Dataset

In [11]:
processed_dfs = []

for symbol in assets:
    asset_df = df.xs(symbol, level=0, axis=1).copy()

    close_price = asset_df['Close']
    volume = asset_df['Volume']
    high = asset_df['High']
    low = asset_df['Low']

    # =========================
    # 1. LOG RETURN (TARGET BASE)
    # =========================
    log_ret = np.log(close_price).diff()
    asset_df["Log_Returns Close"] = log_ret

    # =========================
    # 2. VOLUME STRUCTURE
    # =========================
    sma_volume = ta.SMA(volume, timeperiod=time_prd)
    asset_df["Norm_Volume"] = (
        (volume - sma_volume) / sma_volume
    ).shift(day_shift)

    asset_df["Vol_Shock"] = (
        volume / volume.rolling(time_prd).mean()
    ).shift(day_shift)

    # =========================
    # 3. PRICE DISTANCE
    # =========================
    sma_price = ta.SMA(close_price, timeperiod=time_prd)

    asset_df[f"Distance_SMA_{time_prd}"] = (
        (close_price - sma_price) / close_price
    ).shift(day_shift)

    # =========================
    # 4. REALIZED VOL
    # =========================
    asset_df[f"RV_{time_prd}"] = (
        np.sqrt((log_ret**2).rolling(time_prd).sum())
    ).shift(day_shift)

    # =========================
    # 5. CUMULATIVE RETURNS
    # =========================
    asset_df["CumRet_5"] = (
        np.log(close_price).diff(5)
    ).shift(day_shift)

    asset_df["CumRet_20"] = (
        np.log(close_price).diff(20)
    ).shift(day_shift)

    # =========================
    # 6. RANGE / VOL PROXY
    # =========================
    asset_df["HL_Range"] = (
        (high - low) / close_price
    ).shift(day_shift)

    asset_df[f"ATR_{time_prd}"] = (
        ta.ATR(high, low, close_price, timeperiod=time_prd)
    ).shift(day_shift)

    # =========================
    # 7. BOLLINGER POSITION
    # =========================
    std = close_price.rolling(time_prd).std()

    asset_df["BB_Position"] = (
        (close_price - sma_price) / (2 * std)
    ).shift(day_shift)

    # =========================
    # 8. SEMI VARIANCE
    # =========================
    downside = log_ret.copy()
    downside[downside > 0] = 0

    asset_df[f"SemiVar_{time_prd}"] = (
        np.sqrt((downside**2).rolling(time_prd).mean())
    ).shift(day_shift)

    # =========================
    # 9. VOL REGIME
    # =========================
    rv = asset_df[f"RV_{time_prd}"]

    asset_df["Vol_Regime"] = (
        rv > rv.rolling(60).mean()
    ).astype(int).shift(day_shift)

    # =========================
    # 10. ROLLING VWAP
    # =========================
    typical_price = (high + low + close_price) / 3
    rolling_tp_vol = (typical_price * volume).rolling(time_prd).sum()
    rolling_vol = volume.rolling(time_prd).sum()
    rolling_vwap = rolling_tp_vol / rolling_vol

    asset_df["Distance_Rolling_VWAP"] = (
        (close_price - rolling_vwap) / close_price
    ).shift(day_shift)

    # =========================
    # MultiIndex restore
    # =========================
    asset_df.columns = pd.MultiIndex.from_product([[symbol], asset_df.columns])
    processed_dfs.append(asset_df)

df_updated = pd.concat(processed_dfs, axis=1)
df_updated = df_updated.dropna()

In [12]:
nan_counts = df_updated.isna().sum()
print(nan_counts[nan_counts > 0])

Series([], dtype: int64)


In [13]:
df_updated.dropna(inplace=True)
nan_counts = df_updated.isna().sum()
print(nan_counts[nan_counts > 0])
df_updated

Series([], dtype: int64)


AAPL                                                \
                 Close        High         Low        Open    Volume   
Date                                                                   
2021-02-03  130.510605  132.293751  130.189052  132.283998  89880900   
2021-02-04  133.872253  133.881992  131.143942  132.810165  84183100   
2021-02-05  133.457504  134.101570  132.579243  134.033268  75693800   
2021-02-08  133.603882  133.652677  131.661931  132.745127  71297200   
2021-02-09  132.725632  134.550485  132.569507  133.320902  76774200   
...                ...         ...         ...         ...       ...   
2024-12-24  257.286682  257.296626  254.386957  254.586262  23234700   
2024-12-26  258.103729  259.179926  256.718662  257.276679  27237100   
2024-12-27  254.685867  257.784882  252.164818  256.917934  42355300   
2024-12-30  251.307877  252.603281  249.863009  251.337769  35557500   
2024-12-31  249.534180  252.384064  248.547676  251.547039  39480700   

                                                                              \
           Log_Returns Close Norm_Volume Vol_Shock Distance_SMA_20     RV_20   
Date                                                                           
2021-02-03         -0.007809   -0.271330  0.728670        0.011994  0.101119   
2021-02-04          0.025432   -0.211129  0.788871        0.003154  0.100673   
2021-02-05         -0.003103   -0.237409  0.762591        0.024259  0.098027   
2021-02-08          0.001096   -0.303623  0.696377        0.019020  0.092158   
2021-02-09         -0.006595   -0.333693  0.666307        0.018248  0.091763   
...                      ...         ...       ...             ...       ...   
2024-12-24          0.011413   -0.189446  0.810554        0.040126  0.045300   
2024-12-26          0.003171   -0.506302  0.493698        0.046114  0.044880   
2024-12-27         -0.013331   -0.409495  0.590505        0.044508  0.044007   
2024-12-30         -0.013352   -0.090463  0.909537        0.027644  0.045979   
2024-12-31         -0.007083   -0.242196  0.757804        0.011626  0.046787   

            ...            CSCO                                          \
            ... Distance_SMA_20     RV_20  CumRet_5 CumRet_20  HL_Range   
Date        ...                                                           
2021-02-03  ...        0.014794  0.039737  0.012515  0.041659  0.011128   
2021-02-04  ...        0.011547  0.039756  0.001531  0.039894  0.013546   
2021-02-05  ...        0.039492  0.050029  0.041263  0.062213  0.032804   
2021-02-08  ...        0.052829  0.051469  0.075581  0.067093  0.017471   
2021-02-09  ...        0.065509  0.054392  0.075083  0.082600  0.015938   
...         ...             ...       ...       ...       ...       ...   
2024-12-24  ...        0.000907  0.035201  0.010739  0.007317  0.015259   
2024-12-26  ...        0.014503  0.037987  0.022473  0.018720  0.017544   
2024-12-27  ...        0.016314  0.035232  0.041705  0.006523  0.011004   
2024-12-30  ...        0.009940  0.035414  0.033780  0.005383  0.014763   
2024-12-31  ...        0.002931  0.036087  0.011384 -0.000338  0.016557   

                                                                              
              ATR_20 BB_Position SemiVar_20 Vol_Regime Distance_Rolling_VWAP  
Date                                                                          
2021-02-03  0.720684    0.743457   0.005339        0.0              0.014932  
2021-02-04  0.711616    0.688551   0.005347        0.0              0.011880  
2021-02-05  0.743450    1.700688   0.005347        0.0              0.040068  
2021-02-08  0.753685    1.577145   0.005347        0.0              0.052720  
2021-02-09  0.770802    1.460227   0.005347        0.0              0.064511  
...              ...         ...        ...        ...                   ...  
2024-12-24  0.847663    0.040218   0.005613        0.0              0.004061  
2024-12-26  0.856460    0.624991   0.005613    

In [14]:
drop_cols = ['Open', 'High', 'Low', 'Volume']

df_final = df_updated.drop(columns=drop_cols, level=1).copy()
df_final

AAPL                                          \
                 Close Log_Returns Close Norm_Volume Vol_Shock   
Date                                                             
2021-02-03  130.510605         -0.007809   -0.271330  0.728670   
2021-02-04  133.872253          0.025432   -0.211129  0.788871   
2021-02-05  133.457504         -0.003103   -0.237409  0.762591   
2021-02-08  133.603882          0.001096   -0.303623  0.696377   
2021-02-09  132.725632         -0.006595   -0.333693  0.666307   
...                ...               ...         ...       ...   
2024-12-24  257.286682          0.011413   -0.189446  0.810554   
2024-12-26  258.103729          0.003171   -0.506302  0.493698   
2024-12-27  254.685867         -0.013331   -0.409495  0.590505   
2024-12-30  251.307877         -0.013352   -0.090463  0.909537   
2024-12-31  249.534180         -0.007083   -0.242196  0.757804   

                                                                              \
           Distance_SMA_20     RV_20  CumRet_5 CumRet_20  HL_Range    ATR_20   
Date                                                                           
2021-02-03        0.011994  0.101119 -0.058762  0.042215  0.012594  4.060312   
2021-02-04        0.003154  0.100673 -0.058858  0.022118  0.016127  3.962531   
2021-02-05        0.024259  0.098027  0.002186  0.081791  0.020453  3.932974   
2021-02-08        0.019020  0.092158  0.037222  0.045134  0.011407  3.812442   
2021-02-09        0.018248  0.091763  0.021933  0.037636  0.014900  3.721357   
...                    ...       ...       ...       ...       ...       ...   
2024-12-24        0.040126  0.045300  0.016710  0.104808  0.008618  4.056363   
2024-12-26        0.046114  0.044880  0.018450  0.103254  0.011309  4.000026   
2024-12-27        0.044508  0.044007  0.043275  0.097064  0.009536  3.923088   
2024-12-30        0.027644  0.045979  0.022954  0.084287  0.022067  4.023879   
2024-12-31        0.011626  0.046787 -0.009039  0.060771  0.010904  4.063828   

            ...            CSCO                                          \
            ... Distance_SMA_20     RV_20  CumRet_5 CumRet_20  HL_Range   
Date        ...                                                           
2021-02-03  ...        0.014794  0.039737  0.012515  0.041659  0.011128   
2021-02-04  ...        0.011547  0.039756  0.001531  0.039894  0.013546   
2021-02-05  ...        0.039492  0.050029  0.041263  0.062213  0.032804   
2021-02-08  ...        0.052829  0.051469  0.075581  0.067093  0.017471   
2021-02-09  ...        0.065509  0.054392  0.075083  0.082600  0.015938   
...         ...             ...       ...       ...       ...       ...   
2024-12-24  ...        0.000907  0.035201  0.010739  0.007317  0.015259   
2024-12-26  ...        0.014503  0.037987  0.022473  0.018720  0.017544   
2024-12-27  ...        0.016314  0.035232  0.041705  0.006523  0.011004   
2024-12-30  ...        0.009940  0.035414  0.033780  0.005383  0.014763   
2024-12-31  ...        0.002931  0.036087  0.011384 -0.000338  0.016557   

                                                                              
              ATR_20 BB_Position SemiVar_20 Vol_Regime Distance_Rolling_VWAP  
Date                                                                          
2021-02-03  0.720684    0.743457   0.005339        0.0              0.014932  
2021-02-04  0.711616    0.688551   0.005347        0.0              0.011880  
2021-02-05  0.743450    1.700688   0.005347        0.0              0.040068  
2021-02-08  0.753685    1.577145   0.005347        0.0              0.052720  
2021-02-09  0.770802    1.460227   0.005347        0.0              0.064511  
...              ...         ...        ...        ...                   ...  
2024-12-24  0.847663    0.040218   0.005613        0.0              0.004061  
2024-12-26  0.856460    0.624991   0.005613        0.0              0.018037  
2024-12-27  0.845808    0.681903   0.005613        0

In [15]:
close_price_df = df_final.xs('Close', level=1, axis=1)
close_price_df

,AAPL,TSLA,MSFT,NVDA,GOOGL,AMZN,GOOG,META,AVGO,ORCL,CRM,ADBE,AMD,CSCO
Date,,,,,,,,,,,,,,
2021-02-03,130.510605,284.896667,233.604568,13.493309,102.172020,165.626495,102.800018,264.800354,41.928913,58.210693,232.375610,481.920013,87.889999,39.813793
2021-02-04,133.872253,283.329987,232.652863,13.626689,101.911491,166.550003,102.417633,264.641357,42.419239,59.315540,235.502716,489.380005,87.839996,41.101185
2021-02-05,133.457504,284.076660,232.835480,13.553641,103.658287,167.607498,104.187027,266.240204,42.002819,59.549629,236.403259,492.119995,87.900002,41.823177
2021-02-08,133.603882,287.806671,233.095016,14.399062,103.444397,166.147003,103.934258,264.730743,42.605808,59.090828,236.442825,493.760010,91.470001,42.571262
2021-02-09,132.725632,283.153320,234.344788,14.224045,102.991325,165.250000,103.467445,267.580872,42.779770,59.615166,234.236038,496.049988,90.910004,42.188519
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-24,257.286682,462.279999,436.929138,140.189468,195.344940,229.050003,196.932236,605.839600,237.988037,169.721893,342.748352,447.940002,126.290001,58.345390
2024-12-26,258.103729,454.130005,435.715790,139.899521,194.836945,227.050003,196.463745,601.453369,243.627945,169.989227,340.051605,450.160004,125.059998,58.472118
2024-12-27,254.685867,431.660004,428.177216,136.980164,192.007996,223.750000,193.413620,597.924561,240.043442,167.296021,336.797607,446.480011,125.190002,58.111423


In [16]:
target_feature =  "Log_Returns Close"
def create_sequences(df, feature_name, steps: int):
    target_data = df.xs(feature_name, level=1, axis=1)

    sequence_list = []
    # such as steps = 8, loop 0 day to 7 day (include current date)
    for step in range(steps + 1):
        shifted_data = target_data.shift(-step)
        shifted_data.columns = [f"{col}_Day{step}" for col in shifted_data.columns]
        sequence_list.append(shifted_data)

    result_df = pd.concat(sequence_list, axis=1)
    result_df = result_df.sort_index(axis=1)
    return result_df

df_series_seq= create_sequences(df_final, target_feature, steps_sim)
df_series_seq

,AAPL_Day0,AAPL_Day1,AAPL_Day10,AAPL_Day11,AAPL_Day12,AAPL_Day13,AAPL_Day14,AAPL_Day15,AAPL_Day16,AAPL_Day17,...,TSLA_Day55,TSLA_Day56,TSLA_Day57,TSLA_Day58,TSLA_Day59,TSLA_Day6,TSLA_Day60,TSLA_Day7,TSLA_Day8,TSLA_Day9
Date,,,,,,,,,,,,,,,,,,,,,
2021-02-03,-0.007809,0.025432,-0.008674,0.001233,-0.030252,-0.001112,-0.004060,-0.035402,0.002229,0.052452,...,0.013402,0.011993,-0.046386,-0.014781,-0.025377,0.008463,0.046805,0.005480,-0.024686,0.002421
2021-02-04,0.025432,-0.003103,0.001233,-0.030252,-0.001112,-0.004060,-0.035402,0.002229,0.052452,-0.021115,...,0.011993,-0.046386,-0.014781,-0.025377,0.046805,0.005480,-0.035203,-0.024686,0.002421,-0.013586
2021-02-05,-0.003103,0.001096,-0.030252,-0.001112,-0.004060,-0.035402,0.002229,0.052452,-0.021115,-0.024761,...,-0.046386,-0.014781,-0.025377,0.046805,-0.035203,-0.024686,-0.016636,0.002421,-0.013586,-0.007752
2021-02-08,0.001096,-0.006595,-0.001112,-0.004060,-0.035402,0.002229,0.052452,-0.021115,-0.024761,-0.015938,...,-0.014781,-0.025377,0.046805,-0.035203,-0.016636,0.002421,-0.003957,-0.013586,-0.007752,-0.089376
2021-02-09,-0.006595,-0.004569,-0.004060,-0.035402,0.002229,0.052452,-0.021115,-0.024761,-0.015938,0.010681,...,-0.025377,0.046805,-0.035203,-0.016636,-0.003957,-0.013586,-0.011091,-0.007752,-0.089376,-0.022161
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-24,0.011413,0.003171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-12-26,0.003171,-0.013331,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-12-27,-0.013331,-0.013352,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
new_columns = []
original_assets = df.columns.levels[0]

for col in df_series_seq.columns:
    for asset in original_assets:
        if col.startswith(asset):
            suffix = col.replace(asset, "") # "_Day+0", "_Day+1"
            new_feature_name = f"{target_feature}{suffix}"
            new_columns.append((asset, new_feature_name))
            break

# df_series_seq
multi_index_cols = pd.MultiIndex.from_tuples(new_columns, names=['Asset', 'Feature'])
df_series_seq.columns = multi_index_cols

df_final_updated = pd.concat([df_final, df_series_seq], axis=1)
df_final_updated = df_final_updated.sort_index(axis=1, level=0)

In [18]:
df_final_updated['AAPL'].columns

Index(['ATR_20', 'BB_Position', 'Close', 'CumRet_20', 'CumRet_5',
       'Distance_Rolling_VWAP', 'Distance_SMA_20', 'HL_Range',
       'Log_Returns Close', 'Log_Returns Close_Day0', 'Log_Returns Close_Day1',
       'Log_Returns Close_Day10', 'Log_Returns Close_Day11',
       'Log_Returns Close_Day12', 'Log_Returns Close_Day13',
       'Log_Returns Close_Day14', 'Log_Returns Close_Day15',
       'Log_Returns Close_Day16', 'Log_Returns Close_Day17',
       'Log_Returns Close_Day18', 'Log_Returns Close_Day19',
       'Log_Returns Close_Day2', 'Log_Returns Close_Day20',
       'Log_Returns Close_Day21', 'Log_Returns Close_Day22',
       'Log_Returns Close_Day23', 'Log_Returns Close_Day24',
       'Log_Returns Close_Day25', 'Log_Returns Close_Day26',
       'Log_Returns Close_Day27', 'Log_Returns Close_Day28',
       'Log_Returns Close_Day29', 'Log_Returns Close_Day3',
       'Log_Returns Close_Day30', 'Log_Returns Close_Day31',
       'Log_Returns Close_Day32', 'Log_Returns Close_Day33',


In [19]:
target_drop_cols = ["Log_Returns Close"]
df_seq_final = df_final_updated.drop(columns=target_drop_cols, level=1).copy()

In [20]:
df_seq_final.dropna(inplace=True)
nan_counts = df_updated.isna().sum()
print(nan_counts[nan_counts > 0])
df_seq_final.shape

Series([], dtype: int64)


(924, 1036)

In [21]:
import re

# 1. แยกกลุ่มคอลัมน์เหมือนเดิม
level_1_cols = df_seq_final.columns.get_level_values(1).unique().tolist()
first = ['Close']
others = [c for c in level_1_cols if c != 'Close' and 'Log_Returns' not in c]

# 2. จัดการกลุ่ม Log_Returns ด้วย Natural Sort
log_rets = [c for c in level_1_cols if 'Log_Returns' in c]

# ฟังก์ชันดึงตัวเลขจากชื่อคอลัมน์ (เช่น 'Day10' -> 10)
def extract_day_number(column_name):
    match = re.search(r'Day(\d+)', column_name)
    return int(match.group(1)) if match else 0

# เรียงลำดับตามตัวเลขที่สกัดได้
log_rets_sorted = sorted(log_rets, key=extract_day_number)

# 3. รวมลำดับใหม่และ reindex
new_level_1_order = first + others + log_rets_sorted
df_seq_final = df_seq_final.reindex(columns=new_level_1_order, level=1)
df_seq_final

AAPL                                            \
                 Close    ATR_20 BB_Position CumRet_20  CumRet_5   
Date                                                               
2021-02-03  130.510605  4.060312    0.154294  0.042215 -0.058762   
2021-02-04  133.872253  3.962531    0.040486  0.022118 -0.058858   
2021-02-05  133.457504  3.932974    0.332030  0.081791  0.002186   
2021-02-08  133.603882  3.812442    0.260397  0.045134  0.037222   
2021-02-09  132.725632  3.721357    0.249844  0.037636  0.021933   
...                ...       ...         ...       ...       ...   
2024-09-30  231.920654  4.677131    0.542917 -0.008742 -0.001798   
2024-10-01  225.162094  4.702569    1.080764  0.017316  0.028426   
2024-10-02  225.729446  4.928297    0.264935  0.015324 -0.005115   
2024-10-03  224.624588  4.898374    0.298040  0.026497  0.001810   
2024-10-04  225.749344  4.827146    0.149684  0.014686 -0.008164   

                                                                        \
           Distance_Rolling_VWAP Distance_SMA_20  HL_Range Norm_Volume   
Date                                                                     
2021-02-03              0.009494        0.011994  0.012594   -0.271330   
2021-02-04              0.000453        0.003154  0.016127   -0.211129   
2021-02-05              0.021836        0.024259  0.020453   -0.237409   
2021-02-08              0.016596        0.019020  0.011407   -0.303623   
2021-02-09              0.016148        0.018248  0.014900   -0.333693   
...                          ...             ...       ...         ...   
2024-09-30              0.013607        0.018578  0.009746   -0.446977   
2024-10-01              0.035084        0.039665  0.014378   -0.114650   
2024-10-02              0.005646        0.010079  0.026126    0.016470   
2024-10-03              0.007286        0.011260  0.019182   -0.467190   
2024-10-04              0.002122        0.005668  0.015465   -0.447182   

                      ...                    TSLA                          \
               RV_20  ... Log_Returns Close_Day51 Log_Returns Close_Day52   
Date                  ...                                                   
2021-02-03  0.101119  ...               -0.034588                0.006082   
2021-02-04  0.100673  ...                0.006082                0.034355   
2021-02-05  0.098027  ...                0.034355               -0.033382   
2021-02-08  0.092158  ...               -0.033382                0.013402   
2021-02-09  0.091763  ...                0.013402                0.011993   
...              ...  ...                     ...                     ...   
2024-09-30  0.060514  ...                0.057611               -0.015827   
2024-10-01  0.064510  ...               -0.015827                0.042449   
2024-10-02  0.065386  ...                0.042449                0.059601   
2024-10-03  0.064860  ...                0.059601                0.035724   
2024-10-04  0.064678  ...                0.035724               -0.086424   

                                                            \
           Log_Returns Close_Day53 Log_Returns Close_Day54   
Date                                                         
2021-02-03                0.034355               -0.033382   
2021-02-04               -0.033382                0.013402   
2021-02-05                0.013402                0.011993   
2021-02-08                0.011993               -0.046386   
2021-02-09               -0.046386               -0.014781   
...                            ...                     ...   
2024-09-30                0.042449                0.059601   
2024-10-01                0.059601                0.035724   
2024-10-02                0.035724               -0.086424   
2024-10-03               -0.086424               -0.009038   
2024-10-04               -0.009038               -0.035257   

                                                            \
           Log_Ret

In [22]:
df_seq_final.xs("AAPL", level=0, axis=1).columns.unique()

Index(['Close', 'ATR_20', 'BB_Position', 'CumRet_20', 'CumRet_5',
       'Distance_Rolling_VWAP', 'Distance_SMA_20', 'HL_Range', 'Norm_Volume',
       'RV_20', 'SemiVar_20', 'Vol_Regime', 'Vol_Shock',
       'Log_Returns Close_Day0', 'Log_Returns Close_Day1',
       'Log_Returns Close_Day2', 'Log_Returns Close_Day3',
       'Log_Returns Close_Day4', 'Log_Returns Close_Day5',
       'Log_Returns Close_Day6', 'Log_Returns Close_Day7',
       'Log_Returns Close_Day8', 'Log_Returns Close_Day9',
       'Log_Returns Close_Day10', 'Log_Returns Close_Day11',
       'Log_Returns Close_Day12', 'Log_Returns Close_Day13',
       'Log_Returns Close_Day14', 'Log_Returns Close_Day15',
       'Log_Returns Close_Day16', 'Log_Returns Close_Day17',
       'Log_Returns Close_Day18', 'Log_Returns Close_Day19',
       'Log_Returns Close_Day20', 'Log_Returns Close_Day21',
       'Log_Returns Close_Day22', 'Log_Returns Close_Day23',
       'Log_Returns Close_Day24', 'Log_Returns Close_Day25',
       'Log_Retu

In [23]:
# # type(df_final_updated)
# # df
# df_seq_final.xs("AAPL", level=0, axis=1).columns

# level_1_cols = df_seq_final.columns.get_level_values(1).unique().tolist()

# # 2. จัดกลุ่มตามเงื่อนไขเดิม
# first = ['Close']
# log_rets = [c for c in level_1_cols if 'Log_Returns' in c]
# others = [c for c in level_1_cols if c not in first and 'Log_Returns' not in c]

# # รวมลำดับที่ต้องการ: Close -> Others -> Log_Returns
# new_level_1_order = first + others + log_rets

# # 3. ใช้ reindex เพื่อจัดลำดับใหม่ในระดับ Level 1 สำหรับทุกหุ้น
# df_seq_final = df_seq_final.reindex(columns=new_level_1_order, level=1)
# df_seq_final

In [24]:
n_obs = len(df_seq_final)
n_assets = len(df_seq_final.columns.get_level_values(0).unique())
n_features = len(df_seq_final.columns.get_level_values(1).unique())

print(n_obs, n_assets, n_features)

924 14 74


In [25]:
market = df_seq_final.values.reshape(n_obs, n_assets, n_features)
print(type(market))
market.shape

# tensor_list = []
     #    for symbol in self.symbols:
     #        if symbol in self.assets:
     #            # Call Asset method
     #            asset_tensor = self.assets[symbol].to_tensor(features, device)
     #            tensor_list.append(asset_tensor)
        
     #    # Stack along dimension 1 (Dimension N)
     #    # Asset: [T, F] -> Stack dim=1 -> [T, A, F]
     #    basket_tensor = torch.stack(tensor_list, dim=1)

<class 'numpy.ndarray'>


(924, 14, 74)

In [26]:
df_seq_final.columns.remove_unused_levels()
df_seq_final.columns.get_level_values(1)

Index(['Close', 'ATR_20', 'BB_Position', 'CumRet_20', 'CumRet_5',
       'Distance_Rolling_VWAP', 'Distance_SMA_20', 'HL_Range', 'Norm_Volume',
       'RV_20',
       ...
       'Log_Returns Close_Day51', 'Log_Returns Close_Day52',
       'Log_Returns Close_Day53', 'Log_Returns Close_Day54',
       'Log_Returns Close_Day55', 'Log_Returns Close_Day56',
       'Log_Returns Close_Day57', 'Log_Returns Close_Day58',
       'Log_Returns Close_Day59', 'Log_Returns Close_Day60'],
      dtype='object', length=1036)

In [27]:
df_seq_final['AAPL'].columns

Index(['Close', 'ATR_20', 'BB_Position', 'CumRet_20', 'CumRet_5',
       'Distance_Rolling_VWAP', 'Distance_SMA_20', 'HL_Range', 'Norm_Volume',
       'RV_20', 'SemiVar_20', 'Vol_Regime', 'Vol_Shock',
       'Log_Returns Close_Day0', 'Log_Returns Close_Day1',
       'Log_Returns Close_Day2', 'Log_Returns Close_Day3',
       'Log_Returns Close_Day4', 'Log_Returns Close_Day5',
       'Log_Returns Close_Day6', 'Log_Returns Close_Day7',
       'Log_Returns Close_Day8', 'Log_Returns Close_Day9',
       'Log_Returns Close_Day10', 'Log_Returns Close_Day11',
       'Log_Returns Close_Day12', 'Log_Returns Close_Day13',
       'Log_Returns Close_Day14', 'Log_Returns Close_Day15',
       'Log_Returns Close_Day16', 'Log_Returns Close_Day17',
       'Log_Returns Close_Day18', 'Log_Returns Close_Day19',
       'Log_Returns Close_Day20', 'Log_Returns Close_Day21',
       'Log_Returns Close_Day22', 'Log_Returns Close_Day23',
       'Log_Returns Close_Day24', 'Log_Returns Close_Day25',
       'Log_Retu

In [28]:
# market[0, 0, 0] # -> that close price 


print(f"Market shape:\t\t{market.shape}")
print(f"Close price:\t\t{market[:,:,0:1].shape}")
print(f"Features_Condition:\t{market[:,:,1:n_conditions + 1].shape}")
print(f"Features_Target:\t{market[:,:,n_conditions + 1:].shape}")

print(f"That's a close price:\t{market[0,0,0:1]}")
print(f"Feature_Cond values:\t{market[0, 0, 1:n_conditions + 1]}")
print(f"Feature Target values:\t{market[0, 0, n_conditions + 1:]}")

price = market[:, :, 0:1]
x     = market[:, :, n_conditions + 1:]
cond  = market[:, :, 1:n_conditions + 1]
date  = df_seq_final.index.get_level_values(0).unique().to_numpy()

# print(f"price shape: {price.shape}")
# print(f"x shape: {x.shape}")
# print(f"cond shape: {cond.shape}")
# print(f"date shape: {date.shape}")

Market shape:		(924, 14, 74)
Close price:		(924, 14, 1)
Features_Condition:	(924, 14, 12)
Features_Target:	(924, 14, 61)
That's a close price:	[130.51060486]
Feature_Cond values:	[ 4.06031175  0.15429367  0.04221507 -0.05876218  0.00949441  0.01199355
  0.0125935  -0.27132992  0.10111901  0.01570591  0.          0.72867008]
Feature Target values:	[-0.00780877  0.02543153 -0.00310291  0.00109621 -0.00659524 -0.00456901
 -0.00192212  0.00177428 -0.0162348  -0.01780159 -0.00867391  0.00123269
 -0.03025219 -0.00111167 -0.00406032 -0.035402    0.00222911  0.05245155
 -0.02111518 -0.02476068 -0.01593809  0.01068113 -0.04256686  0.03984534
 -0.00920903  0.01636802 -0.00765465  0.02416245  0.0126624  -0.00647138
 -0.03449327 -0.0044901   0.02794145 -0.00691251 -0.0201958   0.00415468
  0.00512826  0.00148397 -0.01235049  0.01859164  0.00693444  0.0233039
  0.00245905  0.01330142  0.01905115  0.0200495  -0.01332141  0.02401574
 -0.01801454  0.01853538 -0.00253129  0.0050555  -0.01291264  0.0029

In [29]:
def calc_ts_splits(total_samples, ratios, gap):
    total_gaps = gap * 2 # Actually there're 3 gap which one was cutted for NaN
    usable_pool = total_samples - total_gaps

    if usable_pool <= 0:
        return "Gap size more than avaliable data"

    train_count = math.floor(usable_pool * ratios[0])
    val_count = math.floor(usable_pool * ratios[1])
    test_count = usable_pool - train_count - val_count

    # -- Train --
    train_start = 0
    train_end = train_count

    # --- Gap 1 ---
    gap1_start = train_end
    gap1_end = gap1_start + gap

    # --- Val ---
    val_start = gap1_end
    val_end = val_start + val_count

    # --- Gap 2 ---
    gap2_start = val_end
    gap2_end = gap2_start + gap

    # --- Test ---
    test_start = gap2_end
    test_end = total_samples

    print(f"--- Configuration: Total {total_samples}, Gap {gap} ---")
    print(f"Train: [{train_start}:{train_end}]\t({train_count} samples)")
    print(f"Gap 1: [{gap1_start}:{gap1_end}]\t(DROPPED)")
    print(f"Val:   [{val_start}:{val_end}]\t({val_count} samples)")
    print(f"Gap 2: [{gap2_start}:{gap2_end}]\t(DROPPED)")
    print(f"Test:  [{test_start}:{test_end}]\t({test_count} samples)")
    
    return (train_start, train_end), (val_start, val_end), (test_start, test_end)

total_data = len(market)
ratios = [0.8, 0.1, 0.1]
gap = steps_sim

indices   = calc_ts_splits(total_data, ratios, gap)
all_dates = df_seq_final.index.get_level_values(0).unique().to_numpy()
all_prices = market[:, :, 0]

print(f"Dates shape: {all_dates.shape}")
print(f"Prices shape: {all_prices.shape}")
print(f"Market shape: {market.shape}")

--- Configuration: Total 924, Gap 60 ---
Train: [0:643]	(643 samples)
Gap 1: [643:703]	(DROPPED)
Val:   [703:783]	(80 samples)
Gap 2: [783:843]	(DROPPED)
Test:  [843:924]	(81 samples)
Dates shape: (924,)
Prices shape: (924, 14)
Market shape: (924, 14, 74)


In [30]:
(t_s, t_e), (v_s, v_e), (te_s, te_e) = indices

train_part = {
    "x": x[t_s:t_e],
    "cond": cond[t_s:t_e],
    "date": date[t_s:t_e],
    "price": price[t_s:t_e],
}

val_part = {
    "x": x[v_s:v_e],
    "cond": cond[v_s:v_e],
    "date": date[v_s:v_e],
    "price": price[v_s:v_e],
}

test_part = {
    "x": x[te_s:te_e],
    "cond": cond[te_s:te_e],
    "date": date[te_s:te_e],
    "price": price[te_s:te_e],
}

# 3. Print เช็คผล (จะเห็นว่า Total ของ X ทั้ง 3 ก้อน จะไม่เท่ากับ len(market) เพราะหัก Gap ไปแล้ว)
print(f"Train X: {train_part['x'].shape}\nVal X: {val_part['x'].shape}\nTest X: {test_part['x'].shape}")
print("-" * 30)
print(f"Train Cond: {train_part['cond'].shape}\nVal Cond: {val_part['cond'].shape}\nTest Cond: {test_part['cond'].shape}")
print("-" * 30)
print(f"Train Price: {train_part['price'].shape}\nVal Price: {val_part['price'].shape}\nTest Price: {test_part['price'].shape}")
print("-" * 30)
print(f"Train Dates: {train_part['date'].shape}\nVal Dates: {val_part['date'].shape}\nTest Dates: {test_part['date'].shape}")

Train X: (643, 14, 61)
Val X: (80, 14, 61)
Test X: (81, 14, 61)
------------------------------
Train Cond: (643, 14, 12)
Val Cond: (80, 14, 12)
Test Cond: (81, 14, 12)
------------------------------
Train Price: (643, 14, 1)
Val Price: (80, 14, 1)
Test Price: (81, 14, 1)
------------------------------
Train Dates: (643,)
Val Dates: (80,)
Test Dates: (81,)


In [31]:
# scaler_x = StandardScaler()
scaler_cond = StandardScaler()

# Scale X
# x_train_2d = train_part['x'].reshape(train_part['x'].shape[0], -1)
# x_val_2d = val_part['x'].reshape(val_part['x'].shape[0], -1)
# x_test_2d = test_part['x'].reshape(test_part['x'].shape[0], -1)

# scaler_x.fit(x_train_2d)

# train_x_scaled = scaler_x.transform(x_train_2d).reshape(train_part['x'].shape)
# val_x_scaled = scaler_x.transform(x_val_2d).reshape(val_part['x'].shape)
# test_x_scaled = scaler_x.transform(x_test_2d).reshape(test_part['x'].shape)

# inspect(train_x_scaled, "X_scaled - Train Part")
# inspect(val_x_scaled, "X_scaled - Val Part")
# inspect(test_x_scaled, "X_scaled - Test Part")

# Scale Cond
cond_train_2d = train_part['cond'].reshape(train_part['cond'].shape[0], -1)
cond_val_2d = val_part['cond'].reshape(val_part['cond'].shape[0], -1)
cond_test_2d = test_part['cond'].reshape(test_part['cond'].shape[0], -1)

scaler_cond.fit(cond_train_2d)

train_cond_scaled = scaler_cond.transform(cond_train_2d).reshape(train_part['cond'].shape)
val_cond_scaled = scaler_cond.transform(cond_val_2d).reshape(val_part['cond'].shape)
test_cond_scaled = scaler_cond.transform(cond_test_2d).reshape(test_part['cond'].shape)

inspect(train_cond_scaled, "Cond_scaled - Train Part")
inspect(val_cond_scaled, "Cond_scaled - Val Part")
inspect(test_cond_scaled, "Cond_scaled - Test Part")

# Pipeline([('Scaler_X', scaler_x), ('Scaler_Cond', scaler_cond)])

--- Inspecting: Cond_scaled - Train Part ---
------------------------------------
Shape: (643, 14, 12)
Min:   -5.3803
Max:   12.7899
Mean:  0.0000
Std:   1.0000
------------------------------------
--- Inspecting: Cond_scaled - Val Part ---
------------------------------------
Shape: (80, 14, 12)
Min:   -3.3936
Max:   10.5660
Mean:  0.0314
Std:   1.0433
------------------------------------
--- Inspecting: Cond_scaled - Test Part ---
------------------------------------
Shape: (81, 14, 12)
Min:   -4.0164
Max:   17.4689
Mean:  0.3005
Std:   1.8647
------------------------------------


(np.float64(-4.01636828490732),
 np.float64(17.46893568447268),
 np.float64(0.30054431659509256),
 np.float64(1.8646764686741786))

In [32]:

# train_ds = MarketDataset(x=train_x_scaled, cond=train_cond_scaled, date=train_part['date'], price=train_part['price'], window_size=window_size, stride=stride, normalize_window=True)
# val_ds = MarketDataset(x=val_x_scaled, cond=val_cond_scaled, date=val_part['date'], price=val_part['price'], window_size=window_size, stride=stride, normalize_window=True)
# test_ds = MarketDataset(x=test_x_scaled, cond=test_cond_scaled, date=test_part['date'], price=test_part['price'], window_size=window_size, stride=stride, normalize_window=True)


train_ds = MarketDataset(x=train_part['x'], cond=train_cond_scaled, date=train_part['date'], price=train_part['price'], window_size=window_size, stride=stride, normalize_window=True)
val_ds = MarketDataset(x=val_part['x'], cond=val_cond_scaled, date=val_part['date'], price=val_part['price'], window_size=window_size, stride=stride, normalize_window=True)
test_ds = MarketDataset(x=test_part['x'], cond=test_cond_scaled, date=test_part['date'], price=test_part['price'], window_size=window_size, stride=stride, normalize_window=True)

print(f"Num of Windows\nTrain DS: {len(train_ds)}, Val Ds: {len(val_ds)}, Test DS: {len(test_ds)}\n")
print(f"A sample shape from Train DS\n\tx: {train_ds[0]['x'].shape},\n\tcond: {train_ds[0]['cond'].shape}\n\tdates: {len(train_ds[0]['date'])}")

Num of Windows
Train DS: 614, Val Ds: 51, Test DS: 52

A sample shape from Train DS
	x: (61, 30, 14),
	cond: (12, 30, 14)
	dates: 30


In [33]:
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size_exp, shuffle=False)

batch_train = next(iter(train_loader))
print(len(train_loader))
print(batch_train["x"].shape)
print(batch_train["cond"].shape)
print(batch_train["x"][0,:,0,0])

10
torch.Size([64, 61, 30, 14])
torch.Size([64, 12, 30, 14])
tensor([ 0.7399,  0.1957, -0.0468,  1.3562,  0.9989,  1.3342, -0.7747,  0.9782,
        -0.4030,  0.5349,  1.8540, -0.3745, -1.1982, -2.4268,  2.0263, -0.6012,
         0.7421,  0.9712,  0.1502, -1.5514, -1.3396,  0.2377,  0.0580, -0.2639,
         0.9677, -0.2639,  0.0233, -0.3644,  0.0300, -0.2081,  0.1275,  1.4956,
         0.1418,  1.0846, -0.3010, -1.7750,  0.2663,  0.8471,  0.9082,  0.1002,
        -0.4879, -0.3168,  0.5953,  2.2787, -0.4665,  0.5584,  0.7219,  0.4744,
         1.2913, -0.6359, -0.4140, -2.4886,  0.3382, -0.8070,  0.5070, -0.1365,
        -1.5695, -1.8208,  0.1755,  1.3011,  0.4736])


In [34]:
len(test_loader)

52

In [35]:
next(iter(test_loader))["x"].shape

torch.Size([1, 61, 30, 14])

In [36]:
batch_train["x"][0,:,0,0]
inv_batch_train_x= inverse_transform(batch_train["x"],batch_train["x_mean"], batch_train["x_std"])
inv_batch_train_x[0,:,0,0]

tensor([ 0.0114,  0.0046,  0.0023,  0.0194,  0.0146,  0.0178, -0.0092,  0.0130,
        -0.0042,  0.0079,  0.0238, -0.0045, -0.0142, -0.0273,  0.0256, -0.0051,
         0.0096,  0.0119,  0.0029, -0.0150, -0.0123,  0.0045,  0.0015, -0.0023,
         0.0126, -0.0028,  0.0007, -0.0048, -0.0003, -0.0034,  0.0018,  0.0206,
         0.0014,  0.0135, -0.0062, -0.0258,  0.0023,  0.0101,  0.0102, -0.0006,
        -0.0085, -0.0055,  0.0072,  0.0300, -0.0085,  0.0045,  0.0074,  0.0042,
         0.0154, -0.0101, -0.0067, -0.0337,  0.0039, -0.0096,  0.0061, -0.0016,
        -0.0185, -0.0216,  0.0034,  0.0167,  0.0067])

In [37]:
B, C_cond, T, A = batch_train["cond"].shape
B, C_target, T, A = batch_train["x"].shape
print(f"B: {B}, C_target: {C_target}, C_cond: {C_cond}, T: {T}, A: {A}")

B: 64, C_target: 61, C_cond: 12, T: 30, A: 14


# Experimental

## Functions

In [38]:
from scipy.stats import skew, kurtosis

def expand_mc_size(x: torch.Tensor, cond: torch.Tensor, n_samples: int):
    return x.expand(n_samples, -1, -1, -1), cond.expand(n_samples, -1, -1, -1)

def transform_dates(dates: torch.Tensor, extend_days: int = 0) -> pd.DatetimeIndex:
    dates_np = dates.cpu().numpy()
    dates_pd = pd.to_datetime(dates_np.flatten(), unit='ns')


    extend_days: int = 0
    if extend_days > 0:
        last_date = dates_pd[-1]
        extended_range = pd.date_range(
            start=last_date + pd.Timedelta(days=1), 
            periods=extend_days, 
            freq='D'
        )
        dates_pd = dates_pd.union(extended_range)
        
    return dates_pd
    
def extract_simulation_stats(sim_data: np.ndarray, method: str = 'separate'):
    """
    สกัดค่า Mu และ Sigma จากข้อมูล Simulation หลายเส้นทาง
    
    Parameters:
    - sim_data: numpy array รูปแบบ [N_sims, Steps, Assets] (เช่น 1000, 20, 5)
    - method: 'combined' (เทรวมกัน) หรือ 'separate' (คิดแยกทีละเส้นทางแล้วเฉลี่ย)
    
    Returns:
    - mu_final: numpy array รูปแบบ [Assets,]
    - sigma_final: numpy array รูปแบบ [Assets, Assets]
    """
    N_sims, Steps, Assets = sim_data.shape

    if method == 'combined':
        # วิธีที่ 1: รวมทุกเส้นทางเป็น Distribution เดียว
        flattened_data = sim_data.reshape(-1, Assets) 
        mu_final = calc_expected_returns(flattened_data)
        sigma_final = calc_covariance(flattened_data)
        
    elif method == 'separate':
        # วิธีที่ 2: คิดแยกทีละเส้นทาง (Path-by-Path)
        mu_list = []
        sigma_list = []
        
        for i in range(N_sims):
            path_data = sim_data[i, :, :] # ดึงมาทีละ 1 เส้นทาง
            
            # เรียกใช้ฟังก์ชันย่อย
            mu_path = calc_expected_returns(path_data)
            sigma_path = calc_covariance(path_data)
            
            mu_list.append(mu_path)
            sigma_list.append(sigma_path)
            
        # นำค่าสถิติของทั้ง 1,000 เส้นทางมาหาค่าเฉลี่ย
        mu_final = np.mean(mu_list, axis=0)
        sigma_final = np.mean(sigma_list, axis=0)
        
    else:
        raise ValueError("Parameter 'method' ต้องเป็น 'combined' หรือ 'separate' เท่านั้น")

    return mu_final, sigma_final


def get_generation_level_stats(data: np.ndarray, model_name: str, is_gt: bool = False) -> pd.DataFrame:
    """
    ดึงค่าสถิติ Model Performance ระดับ Generation Level
    
    Parameters:
    - data: หากเป็น GenAI/Stat จะเป็น shape [N_paths, Steps, Assets]
            หากเป็น Ground Truth (GT) จะเป็น shape [Steps, Assets]
    - model_name: ชื่อโมเดล เช่น 'GenAI', 'GBM-Stat', 'Ground-Truth'
    - is_gt: Flag ระบุว่าเป็น Ground Truth หรือไม่
    
    Returns:
    - pd.DataFrame สรุปค่าสถิติของแต่ละ Asset
    """
    # ถ้าเป็นข้อมูลจำลอง (มี N_paths) เราจะรวบทุกเส้นทางมาดู Distribution รวม
    if not is_gt and len(data.shape) == 3:
        N_paths, Steps, Assets = data.shape
        data_flat = data.reshape(-1, Assets) # รวมเป็น [N_paths * Steps, Assets]
    else:
        # ถ้าเป็น GT จะมีแค่ 2 มิติ [Steps, Assets] อยู่แล้ว
        Assets = data.shape[-1]
        data_flat = data

    stats_list = []
    
    # คำนวณสถิติทีละ Asset
    for asset_idx in range(Assets):
        asset_data = data_flat[:, asset_idx]
        
        stats = {
            'Model': model_name,
            'Asset_Idx': asset_idx,
            'Mean': np.mean(asset_data),
            'Std (Volatility)': np.std(asset_data),
            'Min': np.min(asset_data),
            'Max': np.max(asset_data),
            'Skewness': skew(asset_data),      # ความเบ้ (ดูว่าผลตอบแทนเอียงไปทางบวกหรือลบ)
            'Kurtosis': kurtosis(asset_data)   # ความโด่ง (ดูความเสี่ยงของหาง (Fat-tail risk))
        }
        stats_list.append(stats)
        
    return pd.DataFrame(stats_list)

def calc_annualized_cagr(returns_series, trading_days_per_year=252):
    """คำนวณ CAGR จาก Series ของ Simple Returns"""
    if len(returns_series) == 0: return 0.0
    cum_return = (1 + returns_series).prod()
    return (cum_return ** (trading_days_per_year / len(returns_series))) - 1

In [39]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

# ==========================================
# 1. กราฟแบบใหม่: ใช้ Percentile Bands แทนการพล็อตทุกเส้น
# ==========================================
def plot_simulation_paths(history: np.ndarray, future_gt: np.ndarray, 
                          sim_genai: np.ndarray, sim_gbm: np.ndarray, 
                          asset_idx: int = 0, max_paths: int = 1000,
                          zoom_future: bool = True, anchor_steps: int = 3,
                          save_path: str = None): # 🌟 เพิ่มพารามิเตอร์ save_path
    
    hist_asset = history[:, asset_idx]
    gt_fut_asset = future_gt[:, asset_idx]
    
    plot_genai = sim_genai[:max_paths, :, asset_idx]
    plot_gbm = sim_gbm[:max_paths, :, asset_idx]
    
    H = len(hist_asset)
    F = len(gt_fut_asset)
    start_idx = max(0, H - anchor_steps) if zoom_future else 0
        
    x_hist = np.arange(start_idx, H)
    x_fut = np.arange(H - 1, H + F) 
    
    hist_to_plot = hist_asset[start_idx:]
    last_hist_val = hist_asset[-1]
    gt_fut_line = np.concatenate([[last_hist_val], gt_fut_asset])
    
    mean_genai_line = np.concatenate([[last_hist_val], np.mean(plot_genai, axis=0)])
    mean_gbm_line = np.concatenate([[last_hist_val], np.mean(plot_gbm, axis=0)])
    
    genai_p05 = np.concatenate([[last_hist_val], np.percentile(plot_genai, 5, axis=0)])
    genai_p25 = np.concatenate([[last_hist_val], np.percentile(plot_genai, 25, axis=0)])
    genai_p75 = np.concatenate([[last_hist_val], np.percentile(plot_genai, 75, axis=0)])
    genai_p95 = np.concatenate([[last_hist_val], np.percentile(plot_genai, 95, axis=0)])
    
    gbm_p05 = np.concatenate([[last_hist_val], np.percentile(plot_gbm, 5, axis=0)])
    gbm_p25 = np.concatenate([[last_hist_val], np.percentile(plot_gbm, 25, axis=0)])
    gbm_p75 = np.concatenate([[last_hist_val], np.percentile(plot_gbm, 75, axis=0)])
    gbm_p95 = np.concatenate([[last_hist_val], np.percentile(plot_gbm, 95, axis=0)])
    
    color_history = '#2C3E50'
    color_gt      = '#06C755'
    color_genai   = '#007AFF'
    color_gbm     = '#FF9500'
    
    fig, ax = plt.subplots(figsize=(14, 7), facecolor='white')
    ax.set_facecolor('#FAFAFA') 
    
    plt.fill_between(x_fut, genai_p05, genai_p95, color=color_genai, alpha=0.1, zorder=2)
    plt.fill_between(x_fut, genai_p25, genai_p75, color=color_genai, alpha=0.15, zorder=2)
    
    plt.fill_between(x_fut, gbm_p05, gbm_p95, color=color_gbm, alpha=0.3, zorder=3)
    plt.fill_between(x_fut, gbm_p25, gbm_p75, color=color_gbm, alpha=0.4, zorder=3)
        
    plt.plot(x_fut, mean_genai_line, color=color_genai, linewidth=2, linestyle='-.', zorder=4)
    plt.plot(x_fut, mean_gbm_line, color=color_gbm, linewidth=3, linestyle='--', zorder=5)
        
    plt.plot(x_hist, hist_to_plot, color=color_history, linewidth=2.5, zorder=5)
    plt.plot(x_fut, gt_fut_line, color='#FFFFFF', linewidth=6.5, zorder=6)
    plt.plot(x_fut, gt_fut_line, color=color_gt, linewidth=3.0, zorder=7)
    
    plt.axvline(x=H-1, color='#FF3B30', linestyle='--', alpha=0.8, zorder=1)

    custom_lines = [
        Line2D([0], [0], color=color_history, lw=2.5),
        Line2D([0], [0], color=color_gt, lw=3.0), 
        Line2D([0], [0], color=color_genai, lw=4, alpha=0.5), 
        Line2D([0], [0], color=color_gbm, lw=4, alpha=0.5),   
    ]
    plt.legend(custom_lines, ['History', 'Ground Truth', 'GenAI (5th-95th %ile)', 'GBM (5th-95th %ile)'], 
               loc='upper left', framealpha=0.9, edgecolor='#DDDDDD')
    
    title_suffix = f"(Zoomed Last {anchor_steps} Steps)" if zoom_future else "(Full History)"
    plt.title(f"Simulation Comparison: GenAI vs GBM - Asset {asset_idx} {title_suffix}", fontsize=16, fontweight='bold', pad=15)
    plt.xlabel("Time Steps", fontsize=12, labelpad=10)
    plt.ylabel("Log Return", fontsize=12, labelpad=10)
    
    plt.grid(color='#E0E0E0', linestyle='-', linewidth=0.5, zorder=0)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    
    # 🌟 เซฟรูปลงโฟลเดอร์แบบชัดเป๊ะ (DPI 300)
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"💾 บันทึกรูปกราฟ Asset {asset_idx} ไว้ที่: {save_path}")
        
    plt.show()
    plt.close(fig) # ป้องกันแรมเต็มเมื่อวนลูปเซฟหลายรูป

In [40]:
import quantstats as qs
def evaluation(dataloader, engine, num_samples_sim: int = 1000, is_dynamic: bool = False, rebalance_freq: int = 5, report_name: str = "portfolio_report"):
    """
    is_dynamic: หากเป็น False คือ Static (จัดครั้งเดียว) | หากเป็น True คือจัดพอร์ตแบบ Rolling
    rebalance_freq: จำนวนวันที่จะถือพอร์ตก่อนทำการจำลองและ Optimize ใหม่ (ใช้เมื่อ is_dynamic=True)
    """
    eval_dir = os.path.join(REPORTS_QS_DIR, f"eval_{checkpoint_filename[:-3]}")
    os.makedirs(eval_dir, exist_ok=True)
    
    count = 0
    
    for batch in dataloader:
        print(f"\n{'='*40}\n🚀 ประมวลผล Batch ที่ {count}\n{'='*40}")
        
        # 🌟 สร้างโฟลเดอร์สำหรับรอบนี้เตรียมไว้ก่อนเลย เพื่อให้พร้อมเซฟไฟล์ทุกชนิด!
        report_dir = os.path.join(eval_dir, f"bt{count}")
        os.makedirs(report_dir, exist_ok=True)
        
        # -----------------------------------------------------
        # 1. GENERATION LEVEL: จำลองข้อมูล
        # -----------------------------------------------------
        x_mc, cond_mc = expand_mc_size(batch['x'], batch['cond'], num_samples_sim)
        sim_full_scaled, sim_only_scaled = engine.simulate(x_mc.to(device), cond_mc.to(device))

        x_mean_adj = batch['x_mean'][:, :1, ...] 
        x_std_adj = batch['x_std'][:, :1, ...]

        sim_only = inverse_transform(sim_only_scaled.to(device), x_mean_adj.to(device), x_std_adj.to(device))

        print(f"🧐 [Debug] Raw GenAI (Scaled) Min: {sim_only_scaled.min().item():.4f} | Max: {sim_only_scaled.max().item():.4f}")
        print(f"🧐 [Debug] After Unscaled Min/Max: {sim_only.min().item():.4f} | {sim_only.max().item():.4f}")
        
        mc_sim_genai = sim_only[:, :, -1:, :].squeeze(2) # [1000, Steps, Assets]
        
        x_unscaled = inverse_transform(batch['x'], batch['x_mean'], batch['x_std'])
        close_log_returns = x_unscaled[0, 0, :, :] 
        steps_sim = mc_sim_genai.shape[1]
        mc_sim_stat = monte_carlo_statistic(torch.as_tensor(close_log_returns), n_sims=num_samples_sim, steps=steps_sim)
        
        gt = x_unscaled[0, 1:, -1:, :].squeeze(1) # [Steps, Assets]
        dates = transform_dates(batch['date'], steps_sim)[-steps_sim:]

        sim_genai_np = mc_sim_genai.detach().cpu().numpy()
        sim_stat_np = mc_sim_stat.detach().cpu().numpy()
        gt_np = gt.detach().cpu().numpy()
        history_np = close_log_returns.detach().cpu().numpy()

        # -----------------------------------------------------
        # 2. MODEL PERFORMANCE: ดึงตารางสถิติ & พล็อตกราฟ 
        # -----------------------------------------------------
        print("\n📊 Extracting Model Statistics...")
        
        df_stats_genai = get_generation_level_stats(sim_genai_np, model_name="GenAI")
        df_stats_stat  = get_generation_level_stats(sim_stat_np, model_name="GBM-Stat")
        df_stats_gt    = get_generation_level_stats(gt_np, model_name="GroundTruth", is_gt=True)

        df_performance = pd.concat([df_stats_gt, df_stats_stat, df_stats_genai], ignore_index=True)
        df_performance['Model'] = pd.Categorical(df_performance['Model'], categories=['GroundTruth', 'GBM-Stat', 'GenAI'], ordered=True)

        # 🌟 บันทึกสถิติระดับ Generation ลงไฟล์ CSV ให้ใช้อ้างอิงใน Paper ได้
        csv_path = os.path.join(report_dir, "generation_level_stats.csv")
        df_performance.to_csv(csv_path, index=False)
        print(f"✅ บันทึกตารางสถิติทุก Asset (CSV) ไว้ที่: {csv_path}")

        num_assets = sim_genai_np.shape[2] 
        print(f"\n{'='*50}\n🔍 เริ่มต้นวิเคราะห์และพล็อตกราฟทีละ Asset (ทั้งหมด {num_assets} ตัว)\n{'='*50}")

        for current_asset in range(num_assets):
            df_target = df_performance[df_performance['Asset_Idx'] == current_asset].sort_values('Model')
            print(f"\n👉 Asset Index: {current_asset}")
            print(df_target.to_string(index=False))
            
            # 🌟 กำหนดพาธสำหรับเซฟรูปกราฟทีละ Asset และโยนเข้าฟังก์ชัน plot_simulation_paths
            plot_save_path = os.path.join(report_dir, f"sim_plot_asset_{current_asset}.png")
            plot_simulation_paths(history_np, gt_np, sim_genai_np, sim_stat_np, asset_idx=current_asset, save_path=plot_save_path)

        
        # -----------------------------------------------------
        # 3. PORTFOLIO OPTIMIZATION & BACKTEST
        # -----------------------------------------------------
        risk_free_rate = 0.02 / 252
        portfolio = Portfolio(risk_free_rate, weight_bounds=(0, 1), save_dir=report_dir)

        gt_simple = np.exp(gt_np) - 1 

        if not is_dynamic:
            print("\n💼 โหมด: Static Portfolio (Buy & Hold)")
            
            mu_genai, sigma_genai = extract_simulation_stats(sim_genai_np, method='separate')
            mu_stats, sigma_stats = extract_simulation_stats(sim_stat_np, method='separate')
            
            w_g = portfolio.optimize_weights(mu_genai, sigma_genai, risk_free_rate=risk_free_rate, scipy=True)
            w_s = portfolio.optimize_weights(mu_stats, sigma_stats, risk_free_rate=risk_free_rate, scipy=True)
            
            port_ret_genai_simple = np.sum(gt_simple * w_g, axis=1)
            port_ret_stats_simple = np.sum(gt_simple * w_s, axis=1)
            
            sr_port_genai_simple = pd.Series(port_ret_genai_simple, index=pd.to_datetime(dates))
            sr_port_stats_simple = pd.Series(port_ret_stats_simple, index=pd.to_datetime(dates))
            
            report_path = os.path.join(report_dir, f'{report_name}_static.html')
            
        else:
            print(f"\n🔄 โหมด: Dynamic Portfolio (Rebalance ทุกๆ {rebalance_freq} วัน)")
            port_ret_genai_simple_list = []
            port_ret_stats_simple_list = []
            
            for start_idx in range(0, steps_sim, rebalance_freq):
                end_idx = min(start_idx + rebalance_freq, steps_sim)
                
                curr_sim_g = sim_genai_np[:, start_idx:, :]
                curr_sim_s = sim_stat_np[:, start_idx:, :]
                curr_gt_simple = gt_simple[start_idx:end_idx, :] 
                
                mu_g, cov_g = extract_simulation_stats(curr_sim_g, method='separate')
                mu_s, cov_s = extract_simulation_stats(curr_sim_s, method='separate')
                
                w_g = portfolio.optimize_weights(mu_g, cov_g, risk_free_rate=risk_free_rate, scipy=True)
                w_s = portfolio.optimize_weights(mu_s, cov_s, risk_free_rate=risk_free_rate, scipy=True)
                
                port_ret_genai_simple_list.extend(np.sum(curr_gt_simple * w_g, axis=1))
                port_ret_stats_simple_list.extend(np.sum(curr_gt_simple * w_s, axis=1))
                
            sr_port_genai_simple = pd.Series(port_ret_genai_simple_list, index=pd.to_datetime(dates))
            sr_port_stats_simple = pd.Series(port_ret_stats_simple_list, index=pd.to_datetime(dates))
            
            report_path = os.path.join(report_dir, f'{report_name}_dynamic.html')

        # -----------------------------------------------------
        # 4. สรุปผล (REPORTING)
        # -----------------------------------------------------
        print("\n🚀 Generating QuantStats Report...")
        qs.reports.html(
            returns=sr_port_genai_simple, 
            benchmark=sr_port_stats_simple, 
            output=report_path, 
            title=f'Portfolio Simulation: GenAI vs GBM', 
            rf=risk_free_rate
        )
        print(f"✅ บันทึกรายงาน (HTML) ไว้ที่: {report_path}")

        cagr_genai = calc_annualized_cagr(sr_port_genai_simple)
        cagr_stat = calc_annualized_cagr(sr_port_stats_simple)

        # 🌟 บันทึกผลลัพธ์ CAGR ออกมาเป็น Text File
        summary_txt = f"=== Performance Summary ===\n" \
                      f"Mode: {'Dynamic' if is_dynamic else 'Static'}\n" \
                      f"GenAI Portfolio CAGR: {cagr_genai * 100:.2f}%\n" \
                      f"GBM-Stat Portfolio CAGR: {cagr_stat * 100:.2f}%\n"
        
        summary_path = os.path.join(report_dir, "performance_summary.txt")
        with open(summary_path, "w", encoding="utf-8") as f:
            f.write(summary_txt)
        print(f"✅ บันทึกสรุปผล CAGR (TXT) ไว้ที่: {summary_path}")

        print(f"\n🏆 Performance Summary (CAGR & Metrics):")
        print(f"   ➤ GenAI Portfolio CAGR: {cagr_genai * 100:.2f}%")
        print(f"   ➤ GBM-Stat Portfolio CAGR:  {cagr_stat * 100:.2f}%")
        print("-" * 40)
        
        qs.reports.metrics(sr_port_genai_simple, benchmark=sr_port_stats_simple, mode='basic', display=True)
            
        count += 1
        break

# Testing

In [46]:

model = DiffusionTransformer(
    num_assets          = A,
    num_channels        = C_target,
    num_cond_channels   = C_cond,
    num_layers          = ddpm_transformer['num_layers'],
    num_attention_heads = ddpm_transformer['nhead'],
    seq_length          = T,
    d_model             = ddpm_transformer['d_model'],
    dim_feedforward     = ddpm_transformer['dim_feedforward'],
    dropout             = ddpm_transformer['dropout']
).to(device)

diffusion = Diffusion(
    model=model,
    timesteps=ddpm['timesteps'],
    beta_start=ddpm['beta_start'],
    beta_end=ddpm['beta_end']
).to(device)

optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, cooldown=2, threshold=0.01)

engine = Engine(
    train_loader    = train_loader,
    val_loader      = val_loader,
    model           = diffusion,
    optimizer       = optimizer,
    criterion       = nn.MSELoss(),
    scheduler       = scheduler,
    device          = device,
    checkpoint_dir  = checkpoint_dir,
    checkpoint_filename = checkpoint_filename
)

checkpoint_dir = os.path.join(RESULTS_DIR,"experimental_checkpoints_no_norm")
checkpoint_filename = f"tr{n_trials}_d{ddpm_transformer['d_model']}_dff{ddpm_transformer['dim_feedforward']}_l{ddpm_transformer['num_layers']}_h{ddpm_transformer['nhead']}_t{ddpm['timesteps']}.pt"

input dim x: 854, d_model: 512


2026-02-16 06:10:16,977 - Engine - INFO - Engine initialized on cuda:0
2026-02-16 06:10:16,981 - Engine - INFO - Criterion: MSELoss


## Call Model to Test

In [48]:
engine.checkpoint_dir = os.path.join(RESULTS_DIR,"experimental_checkpoints")
checkpoint_filename = f"optuna_h31_a14.pt"
engine.load_checkpoint(checkpoint_filename)

RuntimeError: Error(s) in loading state_dict for Diffusion:
	size mismatch for model.x_proj.weight: copying a param with shape torch.Size([512, 448]) from checkpoint, the shape in current model is torch.Size([512, 854]).
	size mismatch for model.output_proj.weight: copying a param with shape torch.Size([448, 512]) from checkpoint, the shape in current model is torch.Size([854, 512]).
	size mismatch for model.output_proj.bias: copying a param with shape torch.Size([448]) from checkpoint, the shape in current model is torch.Size([854]).

## Test Scaling

In [42]:
batch_test = next(iter(test_loader))
print(batch_test['x'].shape) # x = [batch size, channels (include history), length, assets]
print(f"min: {batch_test['x'][0, :, -1, 0].min()}, max: {batch_test['x'][0, :, -1, 0].max()}")

x_unscaled = inverse_transform(batch_test['x'], batch_test['x_mean'], batch_test['x_std'])
print(f"min: {x_unscaled[0, :, -1, 0].min()}, max: {x_unscaled[0, :, -1, 0].max()}")

torch.Size([1, 61, 30, 14])
min: -2.920703172683716, max: 2.660025119781494
min: -0.049365587532520294, max: 0.03639514744281769


In [43]:
sim_full_scaled, sim_only_scaled = engine.simulate(batch_test['x'].to(device), batch_test['cond'].to(device))

print(sim_full_scaled.shape) # x = [batch size, channels (include history), length, assets]
print(f"min: {sim_full_scaled[0, :, -1, 0].min()}, max: {sim_full_scaled[0, :, -1, 0].max()}")

sim_full_unscaled = inverse_transform(sim_full_scaled.cpu(), batch_test['x_mean'].cpu(), batch_test['x_std'].cpu())
print(f"min: {sim_full_unscaled[0, :, -1, 0].min()}, max: {sim_full_unscaled[0, :, -1, 0].max()}")

mask: tensor([[1., 1., 1.,  ..., 1., 1., 1.],
        [1., 1., 1.,  ..., 1., 1., 0.],
        [1., 1., 1.,  ..., 1., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]], device='cuda:0')


torch.Size([1, 61, 30, 14])
min: -10.0, max: 10.0
min: -0.17060652375221252, max: 0.17205049097537994


## Eval

In [ ]:
evaluation(test_loader, engine, paths_sim, is_dynamic=True, report_name="norm_v1_scale")

In [ ]:
print(f"🔍 Standard Deviation - GenAI: {np.std(sim_genai_np):.4f} | GBM: {np.std(sim_stat_np):.4f}")